# Step 3: Feature Selection

### Experiment 1: Use PCA 100 features

1. use selected feature, fit xgboost 

2. Optuna to tune

3. cross validation

4. final model; score test set



In [1]:
import os
import numpy as np
import pandas as pd

import optuna

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import f1_score, make_scorer, accuracy_score

import xgboost as xgb
from sklearn.multioutput import MultiOutputClassifier


/Applications/anaconda3/envs/wids-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = os.getcwd()
home_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
data_dir = os.path.join(home_dir, 'data/')

In [3]:
train_cat = pd.read_csv(os.path.join(data_dir, 'intermediate/encoded_train_cat.csv'))
train_quant = pd.read_csv(os.path.join(data_dir, 'intermediate/train_quant_filled.csv'))
train_fcm_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/train_pca_100.csv'))
train_fcm_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/train_pca_500.csv'))

test_cat = pd.read_csv(os.path.join(data_dir, 'intermediate/encoded_test_cat.csv'))
test_quant = pd.read_csv(os.path.join(data_dir, 'intermediate/test_quant_filled.csv'))
test_fcm_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/test_pca_100.csv'))
test_fcm_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/test_pca_500.csv'))

# selected features
features_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/common_features_selected.csv'), header=None)
features_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/common_features_selected_500.csv'), header=None)

train_label = pd.read_excel(os.path.join(data_dir, 'TRAIN_NEW/TRAINING_SOLUTIONS.xlsx'))

In [4]:
# Join data
X = train_cat.merge(train_quant, on='participant_id', how='left').merge(train_fcm_100, on='participant_id', how='left')
# X = X.drop(columns=['participant_id'], axis=1)
y = train_label.copy()
# y = y.drop(columns=['participant_id'], axis=1)

In [5]:
features_100_list = features_100[0].tolist()
features_100_list = [x.removesuffix('_scaled') for x in features_100_list]

In [7]:
# only keep the selected features
X = X[features_100_list + ['participant_id']]

In [8]:
# merge test data
X_test = test_cat.merge(test_quant, on='participant_id', how='left').merge(test_fcm_100, on='participant_id', how='left')

test_participant_id = X_test['participant_id']

X_test['MRI_Track_Scan_Location_3.0'] = 0

X_test = X_test[features_100_list]

Try scaling the features

In [9]:
# 1. Select numeric columns to scale
numeric_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

# 2. Initialize the scaler
scaler = StandardScaler()

# 3. Fit the scaler on training data only
scaler.fit(X[numeric_features])

# 4. Transform both train and val
X_scaled = X.copy()
X_test_scaled = X_test.copy()

X_scaled[numeric_features] = scaler.transform(X[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

In [10]:
# Split training and test data
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

In [11]:
X_train = X_train.drop(columns=['participant_id'], axis=1)
y_train = y_train.drop(columns=['participant_id'], axis=1)

In [12]:
# Custom scoring function for weighted F1 score
def weighted_f1(y_true, y_pred):
    # Ensure that y_true and y_pred are numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    f1_adhd = f1_score(y_true[:, 0], y_pred[:, 0])
    f1_sex = f1_score(y_true[:, 1], y_pred[:, 1])
    return (2/3) * f1_adhd + (1/3) * f1_sex

In [13]:
y_train

,ADHD_Outcome,Sex_F
215,1,0
482,1,0
27,1,1
1169,0,0
762,1,0
...,...,...
1044,0,1
1095,0,0
1130,0,1
860,0,0


In [16]:
# Objective function for Optuna optimization
def objective(trial):
    # Define the hyperparameters to tune
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 10),
        'random_state': 42,
        'n_jobs': -1,
        'scale_pos_weight': 0.46
    }

    # Define the base model
    base_model = xgb.XGBClassifier(**params)

    # Wrap it with MultiOutputClassifier
    multi_model = MultiOutputClassifier(base_model)

    # Perform cross-validation (here we're using 5-fold cross-validation)
    scores = cross_val_score(multi_model, X_train, y_train, cv=5, scoring=make_scorer(weighted_f1))

    # Return the mean of the cross-validation scores
    return np.mean(scores)


In [17]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print("Best params:", study.best_trial.params)

[I 2025-04-28 13:34:31,094] A new study created in memory with name: no-name-5c2622c8-15e3-488a-a8b0-c14c481c4c6d
[I 2025-04-28 13:34:34,715] Trial 0 finished with value: 0.5087921490124172 and parameters: {'max_depth': 3, 'learning_rate': 0.20485432961364955, 'n_estimators': 492, 'subsample': 0.7569136974224828, 'colsample_bytree': 0.945283541001511, 'gamma': 1.608258745746014}. Best is trial 0 with value: 0.5087921490124172.
[I 2025-04-28 13:34:35,877] Trial 1 finished with value: 0.4660258907453982 and parameters: {'max_depth': 19, 'learning_rate': 0.1312583139660052, 'n_estimators': 58, 'subsample': 0.6948112771657976, 'colsample_bytree': 0.9616369516091042, 'gamma': 4.297508011113505}. Best is trial 0 with value: 0.5087921490124172.
[I 2025-04-28 13:34:39,001] Trial 2 finished with value: 0.48361912916786576 and parameters: {'max_depth': 14, 'learning_rate': 0.1092093077010467, 'n_estimators': 445, 'subsample': 0.7706137136045297, 'colsample_bytree': 0.7806301510162493, 'gamma': 2

Best params: {'max_depth': 11, 'learning_rate': 0.1630961562110346, 'n_estimators': 332, 'subsample': 0.6659706776375881, 'colsample_bytree': 0.773801675255497, 'gamma': 0.0404296614070354}


In [18]:
params = study.best_trial.params
params 


{'max_depth': 11,
 'learning_rate': 0.1630961562110346,
 'n_estimators': 332,
 'subsample': 0.6659706776375881,
 'colsample_bytree': 0.773801675255497,
 'gamma': 0.0404296614070354}

In [19]:
# train with the best params
best_model = xgb.XGBClassifier(**params)
best_multi_model = MultiOutputClassifier(best_model)
best_multi_model.fit(X_train, y_train)

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.773801675255497,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              gamma=0.0404296614070354,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1630961562110346,
                                              max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=11,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=332, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=None, ...))

In [20]:
participant_id = X_val['participant_id']
X_val = X_val.drop(columns = 'participant_id')

In [21]:
# score test set
y_pred = best_multi_model.predict(X_val)

In [22]:
predictions_df = pd.DataFrame(
    y_pred,
    columns=['Predicted_ADHD', 'Predicted_Gender']
)

# Combine participant IDs with predictions
result_df = pd.concat([participant_id.reset_index(drop=True), predictions_df], axis=1)

# Print or save the DataFrame
print(result_df)

    participant_id  Predicted_ADHD  Predicted_Gender
0     ExV0vwCIzAWp               1                 0
1     oSlcxZbncT7a               0                 1
2     28mwHUApaS7q               1                 0
3     8E7WIqYsBQBj               1                 0
4     Hn7obzzz4omm               1                 0
..             ...             ...               ...
359   0FUWCjn9YMN1               1                 0
360   d3VfkRYzJJXX               1                 0
361   vRp3pYC1P44F               1                 0
362   ePfFpufOYJFy               0                 0
363   Jz96BoYNDD51               1                 0

[364 rows x 3 columns]


In [23]:
y_val = y_val.drop(columns = 'participant_id')

In [24]:
f1_test = weighted_f1(y_val, predictions_df)
print(f"Weighted F1 score on test set: {f1_test}")

Weighted F1 score on test set: 0.605574812247256


In [25]:
def multi_output_accuracy(y_true, y_pred):
    # Ensure y_true and y_pred are NumPy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Compute accuracy for each target variable and return the mean
    return np.mean([accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])])

In [26]:
accuracy_test = multi_output_accuracy(y_val, predictions_df)
print(f"Multi-output accuracy on test set: {accuracy_test}")

Multi-output accuracy on test set: 0.6469780219780219


## Train on whole dataset and score test set

In [ ]:
X_scaled = X_scaled.drop(columns=['participant_id'], axis=1)
y = y.drop(columns=['participant_id'], axis=1)

In [31]:
# train with the best params
final_model = xgb.XGBClassifier(**params)
final_multi_model = MultiOutputClassifier(final_model)
final_multi_model.fit(X_scaled, y)

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.773801675255497,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              gamma=0.0404296614070354,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1630961562110346,
                                              max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=11,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=332, n_jobs=None,
                                              num_parallel_tree=None,
                                              random_state=None, ...))

In [32]:
# score test data
y_test_pred = final_multi_model.predict(X_test_scaled)
predictions_test_df = pd.DataFrame(
    y_test_pred,
    columns=['ADHD_Outcome', 'Sex_F']
)
# Combine participant IDs with predictions
result_test_df = pd.concat([test_participant_id.reset_index(drop=True), predictions_test_df], axis=1)

result_test_df

,participant_id,ADHD_Outcome,Sex_F
0,Cfwaf5FX7jWK,1,0
1,vhGrzmvA3Hjq,1,1
2,ULliyEXjy4OV,1,0
3,LZfeAb1xMtql,1,0
4,EnFOUv0YK1RG,0,0
...,...,...,...
299,UadZfjdEg7eG,1,0
300,IUEHiLmQAqCi,1,1
301,cRySmCadYFRO,1,0
302,E3MvDUtJadc5,1,0


In [33]:
result_test_df.to_csv(os.path.join(data_dir, 'output/huizi_xgboost_balance.csv'), index=False)